# 可视化你的数据

<img src="./media/data.gif" width="480" height="360">

基于重建后的仿真场景可视化你的动作。

主窗口会回放动作。

右上和右下叠加图像来自数据集。


In [1]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
import numpy as np
from lerobot.common.datasets.utils import write_json, serialize_dict
REPO_NAME = 'datawhale_eai_pnp_xarm7' #pick-and-place
NUM_DEMO = 1 # Number of demonstrations to collect
ROOT = "./demo_data_xarm7" # The root directory to save the demonstrations
dataset = LeRobotDataset('datawhale_eai_pnp_xarm7', root="./demo_data_xarm7") # if youu want to use the example data provided, root = './demo_data_example' instead!

Generating train split: 0 examples [00:00, ? examples/s]

## 加载数据集


In [2]:
import torch

class EpisodeSampler(torch.utils.data.Sampler):
    """
    Sampler for a single episode
    """
    def __init__(self, dataset: LeRobotDataset, episode_index: int):
        from_idx = dataset.episode_data_index["from"][episode_index].item()
        to_idx = dataset.episode_data_index["to"][episode_index].item()
        self.frame_ids = range(from_idx, to_idx)

    def __iter__(self):
        return iter(self.frame_ids)

    def __len__(self) -> int:
        return len(self.frame_ids)

In [3]:
# Select an episode index that you want to visualize
episode_index = 0

episode_sampler = EpisodeSampler(dataset, episode_index)
dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=1,
    batch_size=1,
    sampler=episode_sampler,
)


## 在仿真中可视化数据集


In [4]:
import env_config

[Env Info] 运行环境已就绪，当前工作目录: C:\Users\kewei\lerobot-mujoco-tutorial


In [5]:
from mujoco_env.y_env import SimpleEnv
# xml_path = './asset/example_scene_y.xml'
xml_path = './asset/example_scene_xarm7.xml'
# 这里选择具体场景即可
PnPEnv = SimpleEnv(xml_path, action_type='joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[27] n_qvel:[25] n_qacc:[25] n_ctrl:[8]
 integrator:[IMPLICITFAST]

n_body:[25]
 [0/25] [world] mass:[0.00]kg
 [1/25] [front_object_table] mass:[1.00]kg
 [2/25] [camera] mass:[0.00]kg
 [3/25] [camera2] mass:[0.00]kg
 [4/25] [camera3] mass:[0.00]kg
 [5/25] [link_base] mass:[0.89]kg
 [6/25] [link1] mass:[2.38]kg
 [7/25] [link2] mass:[1.87]kg
 [8/25] [link3] mass:[1.64]kg
 [9/25] [link4] mass:[1.73]kg
 [10/25] [link5] mass:[1.32]kg
 [11/25] [link6] mass:[1.32]kg
 [12/25] [link7] mass:[0.17]kg
 [13/25] [xarm_gripper_base_link] mass:[0.54]kg
 [14/25] [left_outer_knuckle] mass:[0.03]kg
 [15/25] [left_finger] mass:[0.05]kg
 [16/25] [left_inner_knuckle] mass:[0.02]kg
 [17/25] [right_outer_knuckle] mass:[0.03]kg
 [18/25] [right_finger] mass:[0.05]kg
 [19/25] [right_inner_knuckle] mass:[0.02]kg
 [20/25] [camera_wrist] mass:[0.00]kg
 [21/25] [body_obj_mug_5] mass:[0.00]kg
 [2

In [6]:
# Replay mode options:
# - OVERLAY_SOURCE = "dataset": top-right/bottom-right show recorded images.
# - OVERLAY_SOURCE = "sim":     top-right/bottom-right show current simulation camera images.
OVERLAY_SOURCE = "sim"  # "sim" or "dataset"
RENDER_HZ = 60
ACTION_HZ = 20

step = 0
iter_dataloader = iter(dataloader)
PnPEnv.reset()

action = np.zeros(7, dtype=np.float32)
done = False

while PnPEnv.env.is_viewer_alive() and not done:
    # Update action from dataset at ACTION_HZ
    update_action = PnPEnv.env.loop_every(HZ=ACTION_HZ)
    if update_action:
        try:
            data = next(iter_dataloader)
        except StopIteration:
            break

        if step == 0:
            # Reset object pose from the first frame in this episode
            PnPEnv.set_obj_pose(data['obj_init'][0, :3], data['obj_init'][0, 3:])

        # LeRobot action shape in this notebook: [B=1, 7]
        action = data['action'][0].numpy().astype(np.float32)
        _ = PnPEnv.step(action)
        step += 1

        if OVERLAY_SOURCE == "dataset":
            rgb_agent = (data['observation.image'][0].numpy() * 255).astype(np.uint8)
            rgb_ego = (data['observation.wrist_image'][0].numpy() * 255).astype(np.uint8)
            # CHW -> HWC
            PnPEnv.rgb_agent = np.transpose(rgb_agent, (1, 2, 0))
            PnPEnv.rgb_ego = np.transpose(rgb_ego, (1, 2, 0))
            PnPEnv.rgb_side = np.zeros((480, 640, 3), dtype=np.uint8)

        if step >= len(episode_sampler):
            done = True

    # Advance physics every tick (same pattern as collect loop)
    PnPEnv.step_env()

    # Keep viewer responsive. If using sim overlay, refresh camera frames each render tick.
    if PnPEnv.env.loop_every(HZ=RENDER_HZ):
        if OVERLAY_SOURCE == "sim":
            PnPEnv.grab_image()
        PnPEnv.render(teleop=True)

print(f"Replay finished. steps={step}, episode_len={len(episode_sampler)}")



DONE INITIALIZATION
Replay finished. steps=827, episode_len=827


In [7]:
print(dataset.num_episodes, dataset.num_frames)


1 827


In [30]:
# 为了减少卡顿，建议确保系统内存占用充裕，可关闭不必要的后台程序，或考虑使用远程 Linux 环境以减轻 Windows 负担。

In [31]:
PnPEnv.env.close_viewer()

### 【可选】为其他版本保存 Stats.json


In [32]:
stats = dataset.meta.stats
PATH = dataset.root / 'meta' / 'stats.json'
stats = serialize_dict(stats)

write_json(stats, PATH)